# 03 · Verifier calibration  _(owner: M3)_

Score the Verifier against MedAESQA human labels; iterate the prompt until Cohen's κ ≥ 0.6 / agreement ≥ 80%; show confusion matrices.

### Some requirements if a response is faithful or not

Divide the draft into individual clinical claims. Split sentences containing multiple claims.

Compare each claim with the source.

Check especially for changes to:

- Negation: “no evidence of” versus “has.”

- Certainty: “possible” versus “confirmed.”

- Numbers, units, doses, frequency, duration, and dates.

- Who the statement concerns.

- Past, current, planned, and conditional events.

- Associations versus causes.

We can possibility split the draft and source, and label them to a specific claim and see if the draft states the same information as the source.



### Example
Source: “Possible infection. Follow up in two weeks.”

- Draft: “You may have an infection. Return in two weeks.”
    - Judgment: Both claims are supported; no essential omission.

- Draft: “You definitely have an infection. Return in two weeks.”
    - Judgment: The first claim is unsupported because the source establishes only a possibility.

- Draft: “You may have an infection. Return in two days.”
    - Judgment: The follow-up timing is contradicted by the source.

- Draft: “You may have an infection.”
    - Judgment: The claim is supported, but the follow-up instruction is an essential omission.

In [ ]:
Title": "Muscle cramps",
      "abstract": {
        "1": "Muscle cramps are a common problem characterized by a sudden, painful, involuntary contraction of muscle.",
        "2": "These true cramps, which originate from peripheral nerves, may be distinguished from other muscle pain or spasm.",
      },
      "adaptations": {
        "adaptation2": {
          "1": "Muscle cramps are a common problem represented by sudden, painful, involuntary muscle contractions.",
          "2": "These true cramps, coming from nerves outside the brain and spinal cord, may be identifiable from other muscle pains.",
        }
      }

    "question": "What causes muscle spasm?",
    "question_type": "C"

# Rubric

### Human Grader

Decide whether an AI-output is Faithful or not.

Faithful means 
- if all claims from the output supports the source.
- contradicts nothing in the source, if it does then it is unfaithful
- there is none omission in the draft.


### Step 1. Split the draft into different claims

Break each sentence into single claims
- Example: Cramps are common and usually harmless -> Cramps are common, they are usually harmless

### Step 2. Label each draft claim

We label each claim with either **Supported**, or **Unsupported**

- **Supported**: The source states or clearly implies the draft
- **Unsupported**: The source doesn't support the draft, it added additional fact or either invented new information.
    - added facts or invented details,
    - claims the source contradicts,
    - claims that are stronger, weaker or more specific than the source.

#### Each Supported claim should check for:
- Negation: "no evidence of" vs. "has"
- Certainty: "possible/may" vs. "confirmed/definitely"
- Numbers: amounts, units, doses, frequency, duration, dates
- Subject: who the statement is about
- Time: past, current, planned, conditional
- Cause vs. association: "linked to" vs. "causes"

### Step 3. Check for Omissions

Read the source and list any claims the draft doesn't cover, We want every single claims to be listed to avoid missing crucial informations. So mark any draft as nonfaithful if there is Omission

### Step 6. Verdict
faithful = true only if unsupported_claims is empty and omissions is empty. Otherwise faithful = false. One problem of either kind is enough to fail.



### Worked examples
Source: "Possible infection. Follow up in two weeks."

| Draft | unsupported_claims | omissions | faithful |
|---|---|---|---|
| "You may have an infection. Return in two weeks." | [] | [] | true |
| "You definitely have an infection. Return in two weeks." | ["You definitely have an infection" (certainty)] | [] | false |
| "You may have an infection. Return in two days." | ["Return in two days" (duration; source says two weeks)] | [] | false |
| "You may have an infection." | [] | ["Follow up in two weeks"] | false |



### Edge cases
- **Extra explanation** (defining a term, giving advice): Unsupported unless the source says it.
    - The readability will reword the source term, if there isn't a definition in the source, dont include it. Don't include extra claims in the draft.
- **Vague wording** ("some people" for "10% of patients"): Unsupported. It changes the number.
    - Example: "10% of patients" -> "some people"
- **Ambiguous source:** use the most literal reading, and add a note.
    - Example: "Take with food or after meals" could mean either one is fine, or both are required.
- **Reordered or merged sentences:** Reorder the claims the same way you found it, do not change claims or claim ordering.
- **Repeated claims:** list a problem once.




In [12]:
from __future__ import annotations

import sys
sys.path.append("../src")  # notebooks/ sits next to src/

from clearmed.models import DEFAULT_MODEL, call_model

PROMPT = """You are a strict medical faithfulness verifier. You compare a DRAFT (a plain-language rewrite) against the SOURCE facts (ATOMS). Use only the SOURCE. Do not use outside medical knowledge: a claim that is true but absent from the SOURCE is unsupported.

Follow these steps exactly.

STEP 1 - Split BOTH texts into atomic claims. An atomic claim is ONE fact that could be true or false on its own. Never treat a whole sentence as one claim. Split on:
- every adjective or modifier ("a temporary but intense and painful contraction" = 3 claims: temporary, intense, painful)
- every "and", "but", "although", "while", and semicolon
- every "that ..." or "which ..." clause
- every item in a list
Write each claim as a short full sentence with its subject, numbered in order.
Example: "Muscle cramp is a temporary but intense involuntary contraction that can occur in many situations."
S1 Muscle cramp is temporary. S2 Muscle cramp is intense. S3 Muscle cramp is involuntary. S4 Muscle cramp can occur in many situations.
If a sentence has commas, "and", "but" or "that" and you produced only one claim for it, split it again.

STEP 2 - Match and label each DRAFT claim.
- Supported: the SOURCE states it or clearly implies it. Rewording is fine if it means the same thing to a lay reader (involuntary = uncontrollable, laboratory = lab, likelihood = chance).
- Unsupported: everything else. This includes added facts, invented details, claims the SOURCE contradicts, claims that are stronger, weaker, or more specific than the SOURCE, and claims whose meaning changed even though the words look similar (for example "situations" became "reasons"; "frequency" became "amount"). If you would have to guess or use outside knowledge to call a claim Supported, it is Unsupported.

STEP 3 - Change check. A claim is Unsupported if it differs from the SOURCE on any of:
- Negation ("no evidence of" vs. "has")
- Certainty ("seems likely/appears/may" vs. "is/definitely"; "unknown" vs. "uncertain")
- Numbers, units, doses, frequency, duration, dates
- Who or what the statement concerns
- Time (past, current, planned, conditional; "during or soon after exercise" vs. "exercise-related")
- Cause vs. association ("causes", "involves", "linked to")
Also treat extra explanation or advice not in the SOURCE, and vaguer wording that loses a number ("some people" for "10%"), as Unsupported.

STEP 4 - Omissions. Go through the SOURCE claims one by one. For each, decide if the DRAFT covers it. A claim is covered if the DRAFT says it in simpler words with the same meaning.
If a SOURCE claim is NOT covered, apply this test: would a patient who reads only the DRAFT end up believing, deciding, or doing something different from a patient who read the SOURCE?
- If YES, it is an omission. Add it to "omissions".
- If NO, it is an allowed drop. Do not list it.
NEVER droppable (always an omission if missing): numbers, doses, frequency, duration, dates; timing and conditions ("during or soon after exercise", "if", "unless"); hedges and evidence strength ("there is evidence that", "may", "appears", "seems likely", "some cases", "not all"); negations; who the statement applies to; warnings and risks; instructions and actions; causes, mechanisms, and items in a list.
ALLOWED to drop (not an omission): a purely technical modifier whose removal leaves the same everyday meaning (for example "skeletal" in "skeletal muscle muscle contraction"); intensifier or style words ("notoriously", "very"); repeated information; study-design labels that do not change how strong the evidence sounds (for example "field" in "field trials").
If you are unsure which group a drop belongs to, treat it as an omission.

STEP 5 - Reading level. Set "reading_level_ok" to true if a typical adult without medical training could understand the DRAFT on a first read. Set it false if it keeps unexplained medical terms or very long, dense sentences. This does not affect "faithful".

STEP 6 - Verdict. "faithful" is true only if "unsupported_claims" and "omissions" are both empty. Otherwise false. One problem of either kind is enough to fail.

Rules:
- Match claims by meaning, not by position. Reordering or merging sentences is fine as long as every SOURCE claim is still covered and no claim changed.
- List each problem once. Quote the exact draft claim in "unsupported_claims" and the exact source claim in "omissions".
- If the SOURCE is ambiguous, use its most literal reading.
- Output ONLY this JSON object, with no other text, no markdown fences:
{"faithful": bool, "unsupported_claims": [str], "omissions": [str], "reading_level_ok": bool}
"""



def verify(draft: str, atoms: str, model: str = DEFAULT_MODEL) -> str:
    """Return a JSON verdict string comparing `draft` to `atoms`."""
    return call_model(PROMPT, f"DRAFT:\n{draft}\n\nATOMS:\n{atoms}", model=model)  # dont change, for free / local ollama models to avoid api keys


In [13]:
abstract = [
    "Muscle cramp is a temporary but intense and painful involuntary contraction of skeletal muscle that can occur in many different situations.",
    "The causes of, and cures for, the cramps that occur during or soon after exercise remain uncertain, although there is evidence that some cases may be associated with disturbances of water and salt balance, while others appear to involve sustained abnormal spinal reflex activity secondary to fatigue of the affected muscles.",
    "Evidence in favour of a role for dyshydration comes largely from medical records obtained in large industrial settings, although it is supported by one large-scale intervention trial and by field trials involving small numbers of athletes.",
    "Cramp is notoriously unpredictable, making laboratory studies difficult, but experimental models involving electrical stimulation or intense voluntary contractions of small muscles held in a shortened position can induce cramp in many, although not all, individuals.",
    "These studies show that dehydration has no effect on the stimulation frequency required to initiate cramping and confirm a role for spinal pathways, but their relevance to the spontaneous cramps that occur during exercise is questionable.",
    "There is a long history of folk remedies for treatment or prevention of cramps; some may reduce the likelihood of some forms of cramping and reduce its intensity and duration, but none are consistently effective.",
    "It seems likely that there are different types of cramp that are initiated by different mechanisms; if this is the case, the search for a single strategy for prevention or treatment is unlikely to succeed."
]

adaptation = [
    "Muscle cramp is a temporary but intense, painful, uncontrollable muscle contraction that can occur for different reasons.",
    "The causes of and cures for exercise-related cramps remain unknown. However, some cases may be linked with disturbances of water and salt balance, while others may be from constant abnormal spinal reflex activity linked to fatigue of the affected muscles.",
    "Evidence of lack of water comes largely from medical records from large industrial settings. However, it is also supported by one large treatment trial and trials with small groups of athletes.",
    "Cramp is very unpredictable, making lab studies difficult, but experiments with electrical stimulation or intense voluntary contractions of small muscles held in a shortened position can cause cramp in many, but not all, individuals.",
    "These studies show that dehydration does no affect the stimulation amount needed to start cramping and confirm a role for spinal pathways, but their link to the spontaneous exercise-related cramps that occur is questionable.",
    "There is a long history of home remedies for treatment or prevention of cramps. Some may reduce the chance of some forms of cramping and reduce its intensity and length, but none are consistently effective.",
    "There may be different types of cramp caused by different mechanisms; if so, finding a single strategy for prevention or treatment is unlikely to work."
]

for abstract_claim, adaptation_claim in zip(abstract, adaptation):
    print(verify(adaptation_claim, abstract_claim))

{"faithful": false, "unsupported_claims": "Muscle cramp is a temporary but intense, painful, uncontrollable muscle contraction that can occur for different reasons. ", "omissions": "Muscle cramp is a temporary but intense and painful involuntary contraction of skeletal muscle that can occur in many different situations.", "reading_level_ok": true} 

{"faithful": true, "unsupported_claims": [], "omissions": [], "reading_level_ok": true} 

{"faithful": true, "unsupported_claims": [], "omissions": [], "reading_level_ok": true} 

{"faithful": true, "unsupported_claims": [], "omissions": [], "reading_level_ok": true} 

{"faithful": true, "unsupported_claims": [], "omissions": [], "reading_level_ok": true} 

{"faithful": true, "unsupported_claims": [], "omissions": [], "reading_level_ok": true} 

{"faithful": true, "unsupported_claims": [], "omissions": [], "reading_level_ok": true} 

